# Data exploration

In [1]:
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, random_split


CSV_FILE_TRAIN = "data/train.csv"
CSV_FILE_TEST = "data/test.csv"

df = pd.read_csv(CSV_FILE_TRAIN)
df

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C


## Is Sex a factor in the survival rate ?

In [2]:
#  Sex statistics
missing = df['Sex'].isnull().sum()
print(f"Number of null value is {missing}")

Number of null value is 0


In [3]:
# Sex distribution
df.groupby('Sex')['PassengerId'].count()

Sex
female    314
male      577
Name: PassengerId, dtype: int64

In [4]:
# Survival rate per sex
df.groupby('Sex')['Survived'].mean()

Sex
female    0.742038
male      0.188908
Name: Survived, dtype: float64

Conclusion: females have a 74% survival rate, while males have a 18% survival rate.

## Is socio-economic status (through PClass column) a factor ?

In [5]:
#  Pclass statistics
missing = df['Pclass'].isna().sum()
mean = df['Pclass'].mean()
std = df['Pclass'].std()
print(f"Number of null value is {missing}")
print(f"Mean is {mean}")
print(f"Standard deviation is {std}")

Number of null value is 0
Mean is 2.308641975308642
Standard deviation is 0.836071240977049


In [6]:
# Pclass distribution
df.groupby('Pclass')['PassengerId'].count()

Pclass
1    216
2    184
3    491
Name: PassengerId, dtype: int64

In [7]:
# Survival rate per Pclass
df.groupby('Pclass')['Survived'].mean()

Pclass
1    0.629630
2    0.472826
3    0.242363
Name: Survived, dtype: float64

Conclusion: 1st class has a 62% survival rate while 3rd class has a 24% survival rate

Is age a factor in the survival rate ?

In [8]:
#  Age statistics
missing = df['Age'].isna().sum()
mean = df['Age'].mean()
std = df['Age'].std()
print(f"Number of missing value is {missing}")
print(f"Mean is {mean}")
print(f"Standard deviation is {std}")

Number of missing value is 177
Mean is 29.69911764705882
Standard deviation is 14.526497332334042


In [9]:
# Age group distribution
def get_age_group(age):
    if age <= 10:
        return '0-10'
    elif age <= 20:
        return '10-20'
    elif age <= 30:
        return '20-30'
    elif age <= 40:
        return '30-40'
    elif age <= 50:
        return '40-50'
    elif age <= 60:
        return '50-60'
    elif age <= 70:
        return '60-70'
    elif age <= 80:
        return '70-80'
    else:
        return '80+'


# Remove records where age is missing
df_with_age = df[df['Age'].notna()]

# Create new 'age_group' column
df_with_age['age_group'] = df_with_age['Age'].apply(get_age_group)
df_with_age.groupby('age_group')['PassengerId'].count()

/tmp/ipykernel_797/2107776763.py:27: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_with_age['age_group'] = df_with_age['Age'].apply(get_age_group)


age_group
0-10      64
10-20    115
20-30    230
30-40    155
40-50     86
50-60     42
60-70     17
70-80      5
Name: PassengerId, dtype: int64

In [10]:
# Survival rate per age group
df_with_age.groupby('age_group')['Survived'].mean()

age_group
0-10     0.593750
10-20    0.382609
20-30    0.365217
30-40    0.445161
40-50    0.383721
50-60    0.404762
60-70    0.235294
70-80    0.200000
Name: Survived, dtype: float64

Conclusions:
    - children (< 10yo) are more likely to survive
    - elderly people > 60yo have are less likely to survive

As a first iteration, a small model could be created with the following features: sex, pclass, age group.

# Data preparation

In [11]:
# Keep columns relevant to our model
dfnorm = df.loc[:, ['PassengerId', 'Survived', 'Sex', 'Pclass']]

In [12]:
# Set Sex=1 for female and Sex=0 for male
dfnorm['Sex'] = dfnorm['Sex'].map({'female': 1, 'male': 0})

In [13]:
dfnorm

,PassengerId,Survived,Sex,Pclass
0,1,0,0,3
1,2,1,1,1
2,3,1,1,3
3,4,1,1,1
4,5,0,0,3
...,...,...,...,...
886,887,0,0,2
887,888,1,1,1
888,889,0,1,3
889,890,1,0,1


## Load data in Pytorch datasets

In [14]:
class TitanicDataset(Dataset):
    def __init__(self, csv_file, features_cols, label_col=None):
        """
        Initialize Dataset from CSV file
        """
        self.data = pd.read_csv(csv_file)
        self.features_cols = features_cols
        self.label_col = label_col
        self.has_labels = label_col is not None

        # Set Sex=1 for female and Sex=0 for male
        self.data['Sex'] = self.data['Sex'].map({'female': 1, 'male': 0})

        # Convert data into pytorch tensors
        self.features = torch.tensor(self.data[features_cols].values, dtype=torch.float32)
        if self.has_labels:
            self.labels = torch.tensor(self.data[self.label_col].values, dtype=torch.long)

    def __len__(self):
        """
        Retourne le nombre d'échantillons.
        """
        return len(self.data)

    def __getitem__(self, index):
        """
        Retourne un échantillon (features, label) à l'index donné.
        """
        if self.has_labels:
            return self.features[index], self.labels[index]
        else:
            return self.features[index]


features_cols = ['Sex', 'Pclass']
label_col = 'Survived'

# Use 80% of the dataset to train our model
# Remaning 20% are used to validate the model with unseen data (validation dataset)
train_dataset_length_pct = 0.8
val_dataset_length_pct = 1 - train_dataset_length_pct

dataset = TitanicDataset(CSV_FILE_TRAIN, features_cols, label_col)
train_dataset, val_dataset = random_split(dataset, lengths=[train_dataset_length_pct, val_dataset_length_pct])

# Model training

## Create neural network

In [15]:
# Get cpu, gpu or mps device for training.
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using {device} device")


# Define model
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(2, 4),
            nn.ReLU(),
            nn.Linear(4, 2)
        )

    def forward(self, x):
        logits = self.linear_relu_stack(x)
        return logits


model = NeuralNetwork().to(device)
print(model)

Using cpu device
NeuralNetwork(
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=2, out_features=4, bias=True)
    (1): ReLU()
    (2): Linear(in_features=4, out_features=2, bias=True)
  )
)


In [16]:
batch_size = 16
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

In [17]:
def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    # Set the model to training mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        # Compute prediction and loss
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 8 == 0:
            loss, current = loss.item(), batch * batch_size + len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")


def test_loop(dataloader, model, loss_fn):
    # Set the model to evaluation mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    # Evaluating the model with torch.no_grad() ensures that no gradients are computed during test mode
    # also serves to reduce unnecessary gradient computations and memory usage for tensors with requires_grad=True
    with torch.no_grad():
        for X, y in dataloader:
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [18]:
learning_rate = 1e-2
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

epochs = 50
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(train_dataloader, model, loss_fn, optimizer)
    test_loop(val_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 1.052557  [   16/  713]
loss: 0.832374  [  144/  713]
loss: 0.944394  [  272/  713]
loss: 0.742737  [  400/  713]
loss: 0.726448  [  528/  713]
loss: 0.727456  [  656/  713]
Test Error: 
 Accuracy: 42.7%, Avg loss: 0.705291 

Epoch 2
-------------------------------
loss: 0.729238  [   16/  713]
loss: 0.699369  [  144/  713]
loss: 0.683726  [  272/  713]
loss: 0.655938  [  400/  713]
loss: 0.676654  [  528/  713]
loss: 0.651572  [  656/  713]
Test Error: 
 Accuracy: 59.0%, Avg loss: 0.659462 

Epoch 3
-------------------------------
loss: 0.654351  [   16/  713]
loss: 0.664695  [  144/  713]
loss: 0.592687  [  272/  713]
loss: 0.625535  [  400/  713]
loss: 0.665012  [  528/  713]
loss: 0.617610  [  656/  713]
Test Error: 
 Accuracy: 59.0%, Avg loss: 0.642275 

Epoch 4
-------------------------------
loss: 0.620111  [   16/  713]
loss: 0.645828  [  144/  713]
loss: 0.542969  [  272/  713]
loss: 0.610886  [  400/  713]
loss: 0.662270  [  528/ 

# Prepare predictions

In [36]:
def predict(dataloader, model):
    # Set the model to evaluation mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.eval()
    # Initiliaze empty tensor that will contain our predictions
    pred = torch.tensor([])
    with torch.no_grad():
        for X in dataloader:
            logits = model(X)
            batch_pred = logits.argmax(1)
            pred = torch.cat((pred, batch_pred))
    return pred

In [39]:
features_cols = ['Sex', 'Pclass']
test_batch_size=64

test_dataset = TitanicDataset(CSV_FILE_TEST, features_cols)
test_dataloader = DataLoader(test_dataset, batch_size=test_batch_size, shuffle=False)
pred = predict(test_dataloader, model)
pred

tensor([0., 1., 0., 0., 1., 0., 1., 0., 1., 0., 0., 0., 1., 0., 1., 1., 0., 0.,
        1., 1., 0., 0., 1., 0., 1., 0., 1., 0., 0., 0., 0., 0., 1., 1., 0., 0.,
        1., 1., 0., 0., 0., 0., 0., 1., 1., 0., 0., 0., 1., 1., 0., 0., 1., 1.,
        0., 0., 0., 0., 0., 1., 0., 0., 0., 1., 0., 1., 1., 0., 0., 1., 1., 0.,
        1., 0., 1., 0., 0., 1., 0., 1., 0., 0., 0., 0., 0., 0., 1., 1., 1., 0.,
        1., 0., 1., 0., 0., 0., 1., 0., 1., 0., 1., 0., 0., 0., 1., 0., 0., 0.,
        0., 0., 0., 1., 1., 1., 1., 0., 0., 1., 0., 1., 1., 0., 1., 0., 0., 1.,
        0., 1., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 1., 0., 1., 1., 0., 0.,
        0., 0., 0., 0., 0., 0., 1., 0., 0., 1., 0., 0., 1., 1., 0., 1., 1., 0.,
        1., 0., 0., 1., 0., 0., 1., 1., 0., 0., 0., 0., 0., 1., 1., 0., 1., 1.,
        0., 0., 1., 0., 1., 0., 1., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 1.,
        0., 1., 1., 0., 0., 1., 0., 0., 1., 0., 1., 0., 0., 0., 0., 1., 1., 0.,
        1., 0., 1., 0., 1., 0., 1., 0., 